### Text Embedding for Retrieval: 
### Lesson 1: Introduction
We have a sentence or phrase and get its embedding from a pretrained model.

In [50]:
#!pip install -U sentence-transformers

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import random

In below cell, we initialize a specialized bi-encoder model designed by Microsoft.
Internally, this single line of code triggers a complex pipeline that transforms raw text into a fixed-length numerical vector (an embedding).

In [16]:
model = SentenceTransformer('intfloat/e5-small-v2')

In [38]:
query_text="query: feather coat"
doc_text="passage: Fishing bait accessory, feathered jig lure. Unisex, for cod, rockfish, baitfish, and more. Feathers, highest quality, sizes 3/0 and 5/0. P-Line, fishing context."

The normalize_embeddings Parameter:  
If False (Default): You get the direct output of the mean-pooling layer. The vector lengths (magnitudes) will vary. To compare these, you must use Cosine Similarity.  
If True: The library applies $L2$ normalization. Every vector is scaled to have a length of exactly $1$. In this case, Dot Product and Cosine Similarity become mathematically identical.

In [44]:
query_embed=model.encode(query_text,normalize_embeddings=True)
doc_embed=model.encode(doc_text,normalize_embeddings=True)

Since SentenceTransformer abstracts the tokenization process, you can't see the tokens by just calling .encode(). To see exactly how your text is being chopped up, you need to access the underlying tokenizer object. The model maintains a map of tokens to ID. The transformation from a word like "garden" to an ID like 3842 happens through a process called Vocabulary Mapping. 1. This is a Pre-compiled Vocabulary of  ~30,000 most common words and sub-word pieces. Every single one of these was assigned a fixed index or ID.

0: [PAD]
101: [CLS] (The "Start" marker)
102: [SEP] (The "Separator" marker)
3842: garden
4638: shoes
1. Special Tokens: You'll see [CLS] (start of sequence) and [SEP] (end of sequence).
2. Subword Splitting: Words the model doesn't fully recognize might be split. For exampl e, "tokenization" might become token, ##iz, ##ation.

2. The WordPiece Algorithm
The model doesn't just look up whole words. If it sees a word it doesn't know, it breaks it into pieces it does know. This is why you sometimes see ## in your tokens.

Example: "Tokenizing"

Is "tokenizing" in the vocab? No.
Is "token" in the vocab? Yes. (Assign ID for token)
Is "izing" in the vocab? No.
Is "##iz" in the vocab? Yes. (Assign ID)
Is "##ing" in the vocab? Yes. (Assign ID)

Since SentenceTransformer encapsulates
There are two ways to do this: using the SentenceTransformer object you already have, or using the AutoTokenizer from the transformers library.

1. Using the existing model object
The SentenceTransformer class stores the tokenizer inside its first module (the Transformer layer).

**Why IDs matter to the Model**  
The model (the Neural Network) cannot "read" text. It only understands numbers. However, it doesn't even use the ID 3842 for math. Instead, the ID acts as an address.

The model goes to "address" 3842 in its Embedding Matrix and pulls out a vector of 384 floating-point numbers. That vector is what actually enters the Transformer layers.

In [45]:
# Access the tokenizer from the model's first module
tokenizer = model.tokenizer

# 1. See the Token IDs
tokens = tokenizer.encode(query_text)
print(f"Token IDs: {tokens}")

# 2. See the actual String Tokens (the "WordPieces")
tokens_visual = tokenizer.tokenize(query_text)
print(f"Tokens:    {tokens_visual}")

# 3. See the mapping of ID -> Token
for token_id in tokens:
    print(f"{token_id:<8} -> {tokenizer.decode([token_id])}")

Token IDs: [101, 23032, 1024, 15550, 5435, 102]
Tokens:    ['query', ':', 'feather', 'coat']
101      -> [CLS]
23032    -> query
1024     -> :
15550    -> feather
5435     -> coat
102      -> [SEP]


In [46]:
inputs = tokenizer(query_text, padding=True, truncation=True, return_tensors="pt")
print(inputs)
# Output: {'input_ids': tensor([...]), 'attention_mask': tensor([...])}

{'input_ids': tensor([[  101, 23032,  1024, 15550,  5435,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [47]:
score = model.similarity(query_embed, doc_embed)

In [48]:
print(score)

tensor([[0.8395]])


### Lesson 2

In this lesson, we examine how input text quality influences the effectiveness of generated embeddings. Specifically, we analyze the relationship between input text characteristics and embedding performance, where embedding quality can be expressed as a function of 

$E = f(Q_{\text{text}})$ where $Q_{\text{text}}$ denotes the quality of the input text.
We use eCommerce data as a case study due to its distinct structural and semantic properties compared to other domains, including high variability in product descriptions and domain-specific terminology.
Input Data looks like below

"product_title": "Nike Hike Day Pack (24L)"
"Activity": "Hiking & Backpacking" 
"Brand": "Nike"
"Gender": "Unisex" 
"Style": "DJ9678"

**Option A: Attribute Concatenation (Early Fusion)**  
In this approach, the input $S$ is a single string formed by joining the title ($T$) and the set of attributes ($A = \{a_1, a_2, \dots, a_n\}$).$$S = T \oplus \text{" "} \oplus a_1 \oplus \text{" "} \oplus a_2 \dots$$Technical Characteristics:Cross-Attention: Since the transformer processes the entire string, the self-attention mechanism can learn the inter-dependency between attributes and the title (e.g., how the brand "Nike" modifies the context of the title "Running Shoes").Semantic Noise: If the attribute list is long, the "signal" of the title may be diluted, especially if the model (like E5) has a fixed token limit ($L_{max} = 512$).Performance: Generally superior for Large Language Models (LLMs) and Cross-Encoders because they benefit from the shared context.

In eCommerce Information Retrieval (IR), representing a product as a single vector is a challenge of Feature Fusion. We are comparing Early Fusion (concatenation) versus Late Fusion (individual encoding followed by mean pooling).  

**Option A: String Concatenation (Early Fusion)** In this approach, the input $S$ is a single string formed by joining the title ($T$) and the set of attributes ($A = \{a_1, a_2, \dots, a_n\}$)

**Option B: Vector Averaging (Late Fusion)**   
Here, the title and each attribute are encoded into separate vectors $v \in \mathbb{R}^d$, and the final product representation $V_p$ is the centroid of these vectors.$$V_p = \frac{1}{n+1} \left( E(T) + \sum_{i=1}^{n} E(a_i) \right)$$

To prove mathematically that **Concatenation (Early Fusion)** is superior to Averaging (Late Fusion), we must look at the Non-Linearity of the Transformer architecture and the Information Geometry of the vector space.1. The Interaction Term (Self-Attention)The primary mathematical reason is that Transformers are not linear functions. Let the encoding function be $E(x)$. For Late Fusion (Averaging) to be equal to Early Fusion (Concatenation), the model would have to be additive:$$E(T \oplus A) \approx E(T) + E(A)$$However, the core of a Transformer is the Self-Attention mechanism:$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$When you concatenate Title ($T$) and Attributes ($A$), the Attention matrix contains cross-terms $QK^T$ where the Query comes from a Title token and the Key comes from an Attribute token.  

**Early Fusion:** Captures $\text{Attn}(T, A)$. The model can compute how "Nike" (Attribute) changes the vector representation of "Shoes" (Title).

**Late Fusion**: $\text{Attn}(T, T)$ and $\text{Attn}(A, A)$ are calculated in isolation. The cross-term $\text{Attn}(T, A)$ is mathematically zero because the tokens never see each other.

**Loss of Rank and Dimensionality** When you average $n$ vectors, you are essentially finding the centroid in $\mathbb{R}^d$.If $v_1, v_2, \dots, v_n$ are the embeddings of individual attributes, the average is:$$\mathbf{V}_{avg} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{v}_i$$According to the Central Limit Theorem in high-dimensional spaces, averaging multiple independent vectors tends to push the resulting vector toward the mean of the entire manifold (often called the "Common Vector" problem). This reduces the variance of your embeddings:$$\text{Var}(\mathbf{V}_{avg}) < \text{Var}(\mathbf{V}_{concat})$$Lower variance means your products become "indistinguishable" from one another in the vector space, leading to lower precision in retrieval.

Mathematical Verdict: Because $\text{Attention}(T \oplus A) \neq \text{Attention}(T) + \text{Attention}(A)$, concatenation is required to preserve the covariant relationship between product features.

Implementation code (proof):

In [ ]:
#To be developed

### Lesson 3:

Building on the findings from Lesson 2; which established that concatenating titles and attribute values outperforms the averaging of independent attribute embeddings—we now evaluate the optimal formatting for this input. Specifically, we compare: (a) a single concatenated string of all attributes (early fusion), and (b) a late-fusion approach where each 'attribute name - attribute value' pair is encoded separately and subsequently averaged. Mathematically, it can be described as

Following the proof that $E(T \oplus A)$ yields higher retrieval precision than $\frac{1}{n}\sum E(A_i)$, we seek to identify the superior input format. The evaluation focuses on whether a global concatenation of title and attributes (Early Fusion) provides better semantic alignment than a mean-pooled representation of structured $(a_i, v_i)$ pairs (Late Fusion).

Data looks like: 
Early Fusion: passage:[product_title] "Nike Hike Day Pack (24L)" [Activity] "Hiking & Backpacking" [Brand] "Nike" [Gender] "Unisex" [Style"] "DJ9678"
Late Fusion: passage: "product_title": "Nike Hike Day Pack (24L)"   #each gets its own embedding, this time we have representated as (name:val) pair
             "Activity": "Hiking & Backpacking" 
             "Brand": "Nike"
             "Gender": "Unisex"
             "Style": "DJ9678"

The better representation would be in [ATTRIBUTE_NAME] to distiniguish ":" in "passage:" and using brackets like [BRAND] acts as a pseudo-separator that the model learns to treat as a boundary. 

             passage:[product_title] "Nike Hike Day Pack (24L)"   
             passage:[Activity] "Hiking & Backpacking"   
             passage:[Brand] "Nike"  
             passage:[Gender] "Unisex"  
             passage:[Style] "DJ9678"  
Now let's think about what Math says.


Math behind this representation:
Positional Encoding in Transformer: Knowing the Order- Since you are concatenating everything into one string, the model uses Positional Encodings ($P_i$). This is a set of sine/cosine waves added to the "numbers" to tell the model where each word is.$$V_{final} = E(\text{word}) + P(\text{position})$$This ensures the model knows that [BRAND] belongs to Nike because they are physically adjacent in the sequence. If you had [BRAND] Nike [COLOR] Red, the model's attention heads will strongly link Nike to Brand and Red to Color based on their proximity.

If you average the vectors (Option B), the "numbers" for Brand and Nike are added together. Addition is commutative ($A + B = B + A$). You lose the structural relationship.By keeping them in a sequence (Option A) is still wins.
